# **Estimation of freshwater lens thickness (_FWL_) using three different methods**

> **Objective:** Apply three different methods to estimate the depth of the freshwater lens. Based on this estimation, extract the corresponding array for the freshwater zone, calculate basic statistics, and plot the boxplots for each profile using the three methods.

**Piecewise-regression** — fitting straight line models with breakpoints · Autor: Charlie Pilgrim · v1.5.0 · GitHub: [https://github.com/chasmani/piecewise-regression](https://github.com/chasmani/piecewise-regression) · Docs: [https://piecewise-regression.readthedocs.io/en/master/index.html](https://piecewise-regression.readthedocs.io/en/master/index.html) · Paper: [https://joss.theoj.org/papers/10.21105/joss.03859](https://joss.theoj.org/papers/10.21105/joss.03859)


---

### Import Libraries

In [11]:
import sys
import os
import glob


root = os.path.abspath('../../..')  
sys.path.append(root)

import pandas as pd
import numpy as np

from typing import Tuple, Optional, Any, Dict

import piecewise_regression
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import modules.statistics_fwl_estimation as st_fwl

from modules import processing, load, plots, analysis

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 25)

---

### **Method 1:** _Intuitive criterion_ **(IC)** y **Method 2:** _Optimal BIC_ **(BIC)**

1. Load data `selection_fwl_ic_bic_dgh.csv`

In [12]:
df_estimations = pd.read_csv(f'{root}/data/selection_fwl_ic_bic_dgh.csv')
try:
    df_estimations.loc[df_estimations['profile'] == 'LRS65D_YSI_20230827', 'profile'] = 'LRS65D_YSI_20220812'
    df_estimations.rename(columns={'profile': 'ID'}, inplace=True)
except KeyError:
    df_estimations.loc[df_estimations['ID'] == 'LRS65D_YSI_20230827', 'ID'] = 'LRS65D_YSI_20220812'

df_estimations


,ID,total_bp_ic,fwl_bp_ic,vp_fwl_ic,total_bp_bic,fwl_bp_bic,vp_fwl_bic,vp_fwl_dgh,COMENTARIOS,SEC Profile Features,vp_fwl_lrst
0,AW1D_YSI_20230826,6,2,16.317073,3,1,17.169116,NaN,NaN,NaN,6.120
1,AW2D_YSI_20230815,7,2,9.563224,3,1,9.907636,NaN,NaN,NaN,13.103
2,AW5D_YSI_20230824,5,1,14.134225,4,1,13.652858,NaN,NaN,NaN,12.580
3,AW6D_YSI_20230815,5,2,13.216555,2,1,13.561559,NaN,NaN,NaN,13.016
4,AW7D_YSI_20230814,2,1,13.290227,2,1,13.290227,NaN,No está correctamente cortado el perfil,NaN,12.881
5,BW1D_YSI_20230824,3,1,14.125652,3,1,14.125652,NaN,NaN,NaN,NaN
6,BW2D_YSI_20230819,3,1,13.997452,3,1,13.997452,NaN,NaN,NaN,14.038
7,BW3D_YSI_20230818,4,1,11.701564,4,1,11.701564,NaN,NaN,NaN,10.784
8,BW4D_YSI_20230816,2,1,14.358411,2,1,14.358411,NaN,NaN,NaN,14.014
9,BW5D_YSI_20230822,4,2,12.393402,2,1,11.527182,NaN,NaN,NaN,NaN


---

### **Method 3:** _Dupuit-Ghyben-Herzberg_ **(DGH)** — log10 conductivity

1. Functions to implement DGH using log10-transformed conductivity (`x = depth`, `y = log10(SEC)`)

In [13]:
def extract_single_breakpoint_log10(model: Any) -> Tuple[Optional[float], Optional[float],
                                                          Optional[Tuple[float, float]]]:
    """
    Extract the breakpoint (depth) and predicted log10(SEC) from a fitted
    piecewise regression model with exactly 1 breakpoint.

    Convention: x = depth (Vertical Position), y = log10(conductivity).

    Parameters
    ----------
    model : piecewise_regression.main.Fit
        Fitted model with n_breakpoints=1.

    Returns
    -------
    bp_vp : float or None
        Breakpoint depth (Vertical Position in m).
    bp_log10sec : float or None
        Predicted log10(SEC) at the breakpoint depth.
    conf_interval_vp : tuple(float, float) or None
        Confidence interval (lower, upper) for the breakpoint depth.
    """
    try:
        if not hasattr(model, 'best_muggeo') or not model.best_muggeo:
            raise ValueError("The model did not converge or lacks valid breakpoints.")

        estimates = model.best_muggeo.best_fit.estimates
        bp_key = "breakpoint1"
        bp_vp = estimates[bp_key]["estimate"]
        conf_interval_vp = estimates[bp_key]["confidence_interval"]  # (lower, upper)
        bp_log10sec = model.predict(np.array([bp_vp]))[0]

        return bp_vp, bp_log10sec, conf_interval_vp
    except Exception as e:
        print(f"[DEBUG] extract_single_breakpoint_log10 error: {e}")
        return None, None, None


def dgh_fwl_estimation_log10(
    path: str,
    vp_column: str = 'Vertical Position m',
    sec_column: str = 'Corrected sp Cond [µS/cm]',
    log10_sec_column: str = 'log10_Corrected sp Cond [µS/cm]',
    threshold_log10: float = np.log10(27800),
    tolerance: float = 1e-5,
    min_distance: float = 0.01
) -> Tuple[pd.DataFrame, Dict[str, Dict[str, Any]]]:
    """
    Estimate FWL depth via the DGH method using log10-transformed conductivity.

    Splits each profile into low/high conductivity segments using a threshold
    in log10 space, then fits a piecewise regression (1 breakpoint) per segment
    with the correct convention: x = depth, y = log10(conductivity).

    Parameters
    ----------
    path : str
        Directory containing processed CSV files.
    vp_column : str
        Depth column name.
    sec_column : str
        Raw conductivity column name (used for fallback log10 computation).
    log10_sec_column : str
        Log10-transformed conductivity column name.
    threshold_log10 : float
        Threshold in log10 space to split low/high segments. Default log10(27800) ~ 4.444.
    tolerance : float
        Convergence tolerance for piecewise regression.
    min_distance : float
        Minimum distance between breakpoints.

    Returns
    -------
    summary_df : pd.DataFrame
        Breakpoint estimates per well (depth-based).
    models : dict
        Fitted models keyed by well ID with 'low' and 'high' sub-keys.
    """
    results = []
    models: Dict[str, Dict[str, Any]] = {}

    if not os.path.exists(path):
        raise FileNotFoundError(f"The provided path '{path}' does not exist.")

    csv_files = glob.glob(os.path.join(path, "*.csv"))
    print(f"[DEBUG] CSV files found: {len(csv_files)}")

    if not csv_files:
        raise ValueError(f"No CSV files found in the directory '{path}'.")

    for file in csv_files:
        well_id = os.path.splitext(os.path.basename(file))[0]

        try:
            df = pd.read_csv(file)
        except Exception as e:
            print(f"[DEBUG] Could not read '{well_id}': {e}")
            continue

        # Compute log10 column on the fly if missing
        if log10_sec_column not in df.columns:
            if sec_column not in df.columns:
                print(f"[DEBUG] Skipping '{well_id}': neither '{log10_sec_column}' nor '{sec_column}' found.")
                continue
            df[log10_sec_column] = np.log10(df[sec_column].clip(lower=1e-10))
            print(f"[DEBUG] '{well_id}': computed '{log10_sec_column}' on the fly.")

        if vp_column not in df.columns:
            print(f"[DEBUG] Skipping '{well_id}': column '{vp_column}' not found.")
            continue

        df_filtered = df[[vp_column, sec_column, log10_sec_column]].dropna()
        print(f"[DEBUG] '{well_id}': total={len(df)}, after dropna={len(df_filtered)}")

        # Split using threshold in log10 space
        df_low = df_filtered[df_filtered[log10_sec_column] <= threshold_log10]
        df_high = df_filtered[df_filtered[log10_sec_column] > threshold_log10]

        bp_vp_low, bp_log10sec_low, conf_vp_low = (None, None, None)
        bp_vp_high, bp_log10sec_high, conf_vp_high = (None, None, None)
        well_models = {"low": None, "high": None}

        # Process low conductivity segment (x=depth, y=log10_sec)
        if len(df_low) > 1:
            try:
                model_low = piecewise_regression.Fit(
                    df_low[vp_column].values,
                    df_low[log10_sec_column].values,
                    n_breakpoints=1,
                    tolerance=tolerance,
                    min_distance_between_breakpoints=min_distance
                )
                well_models["low"] = model_low
                bp_vp_low, bp_log10sec_low, conf_vp_low = extract_single_breakpoint_log10(model_low)
            except Exception as e:
                print(f"[DEBUG] Low fit failed for '{well_id}': {e}")

        # Process high conductivity segment (x=depth, y=log10_sec)
        if len(df_high) > 1:
            try:
                model_high = piecewise_regression.Fit(
                    df_high[vp_column].values,
                    df_high[log10_sec_column].values,
                    n_breakpoints=1,
                    tolerance=tolerance,
                    min_distance_between_breakpoints=min_distance
                )
                well_models["high"] = model_high
                bp_vp_high, bp_log10sec_high, conf_vp_high = extract_single_breakpoint_log10(model_high)
            except Exception as e:
                print(f"[DEBUG] High fit failed for '{well_id}': {e}")

        models[well_id] = well_models

        results.append({
            "ID": well_id,
            "breakpoint_1 (vp)": bp_vp_low,
            "breakpoint_2 (vp)": bp_vp_high,
            "breakpoint_1 (log10_sec)": bp_log10sec_low,
            "breakpoint_2 (log10_sec)": bp_log10sec_high,
            "confidence_interval_1 (vp)": conf_vp_low,
            "confidence_interval_2 (vp)": conf_vp_high,
        })

    print(f"[DEBUG] Total wells added to results: {len(results)}")

    summary_df = pd.DataFrame(results)
    return summary_df, models

2. Extract the depths of freshwater lenses using log10(SEC).

In [14]:
dgh_df, models = dgh_fwl_estimation_log10(
    path=f'{root}/data/processed_lrs70',
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]',
    log10_sec_column='log10_Corrected sp Cond [µS/cm]',
    threshold_log10=np.log10(27800),  # ~4.444
    tolerance=1e-5,
    min_distance=0.01
)

[DEBUG] CSV files found: 5
[DEBUG] 'LRS70_D_YSI_20220131_processed': total=2049, after dropna=2049
[DEBUG] 'LRS70_D_YSI_20220816_processed': total=1812, after dropna=1812
[DEBUG] 'LRS70_D_YSI_20230822_processed': total=1801, after dropna=1801
[DEBUG] 'LRS70_D_YSI_20251123_processed': total=4591, after dropna=4591
[DEBUG] 'LRS70_D_YSI_R_20250226_processed': total=1008, after dropna=1008
[DEBUG] Total wells added to results: 5


In [15]:
dgh_df["ID"] = dgh_df["ID"].str.replace("_processed", "", regex=False)

dgh_df

,ID,breakpoint_1 (vp),breakpoint_2 (vp),breakpoint_1 (log10_sec),breakpoint_2 (log10_sec),confidence_interval_1 (vp),confidence_interval_2 (vp)
0,LRS70_D_YSI_20220131,8.903042,12.315419,4.002999,4.718846,"(8.227221064871278, 9.578863323825923)","(12.302335886122957, 12.328503089116118)"
1,LRS70_D_YSI_20220816,7.304788,11.414651,3.842655,4.698633,"(7.208088372339513, 7.401488073259654)","(11.408901538672533, 11.420399974557798)"
2,LRS70_D_YSI_20230822,9.479731,13.003838,2.654999,4.702929,"(9.3968196193071, 9.562642588863929)","(12.956959586713822, 13.050715720654306)"
3,LRS70_D_YSI_20251123,7.564211,13.816140,3.653160,4.717331,"(7.4726353360353714, 7.655787513318953)","(13.805806381882938, 13.826473277063752)"
4,LRS70_D_YSI_R_20250226,9.381218,13.877557,3.770453,4.713319,"(9.325970779958826, 9.436465756834831)","(13.870343100456761, 13.884771399721604)"


In [16]:
dgh_df.to_csv(f'{root}/results/dgh_breakpoints/dgh_log10_27800_processed_lrs70.csv', index=False)

---

### **Method 4:** Land Resource Survey Threshold (LRST)

In [17]:
def lrst_fwl_estimation(path: str,
                        vp_column: str,
                        sec_column: str,
                        threshold: float = 1452.9) -> pd.DataFrame:
    """
    Identifica la posición vertical máxima para cada pozo donde la conductividad específica
    corregida es menor o igual al umbral dado (por defecto 1452.9 uS/cm).
    
    Parameters:
        path (str): Ruta del directorio que contiene los archivos CSV de los pozos.
        vp_column (str): Nombre de la columna que contiene los valores de posición vertical.
        sec_column (str): Nombre de la columna que contiene los valores de conductividad.
        threshold (float): Umbral de conductividad para determinar la posición vertical máxima.
        
    Returns:
        pd.DataFrame: DataFrame con las posiciones verticales máximas para cada pozo.
    """
    results = []
    
    if not os.path.exists(path):
        raise FileNotFoundError(f"La ruta proporcionada '{path}' no existe.")
    
    csv_files = glob.glob(os.path.join(path, "*.csv"))
    if not csv_files:
        raise ValueError(f"No se encontraron archivos CSV en el directorio '{path}'.")
    
    for file in csv_files:
        try:
            df = pd.read_csv(file)
        except Exception as e:
            print(f"Error al leer el archivo {file}: {e}")
            continue
        
        if vp_column not in df.columns or sec_column not in df.columns:
            print(f"Columnas requeridas no encontradas en {file}")
            continue
        
        df_filtered = df[[vp_column, sec_column]].dropna()
        df_low_cond = df_filtered[df_filtered[sec_column] <= threshold]
        
        if not df_low_cond.empty:
            max_vp = df_low_cond[vp_column].max()
        else:
            max_vp = None
        
        well_id = os.path.splitext(os.path.basename(file))[0]
        
        if well_id.endswith('_rowdy'):
            well_id = well_id[:-6]
        
        results.append({
            "ID": well_id,
            "vp_fwl_lrst": max_vp
        })
    
    summary_df = pd.DataFrame(results)
    return summary_df

In [18]:
# lrst_df = lrst_df = lrst_fwl_estimation(
#     path=f'{root}/data/rawdy',
#     vp_column='Vertical Position [m]',
#     sec_column='Corrected sp Cond [uS/cm]',
#     threshold=1452.9
# )

# lrst_df

---

### Combine with the rest of the FWL estimates.

In [19]:
# # Merge with dgh_df first
# df_merged = df_estimations.merge(dgh_df[['ID', 'breakpoint_1 (vp)']], on="ID", how="left")
# df_merged["vp_fwl_dgh"] = df_merged["breakpoint_1 (vp)"]
# df_merged.drop(columns=["breakpoint_1 (vp)"], inplace=True)

# # Now merge with lrst_df to add vp_fwl_lrst
# df_merged = df_merged.merge(lrst_df[['ID', 'vp_fwl_lrst']], on="ID", how="left")

# df_merged

In [20]:
# df_merged.to_csv(f'{root}/data/selection_fwl_ic_bic_dgh.csv', 
#                 index=False
#                 )